# 03 — Targets: Returns, Alpha, Volatility & ROE

In [3]:
from pathlib import Path
import time
import yfinance as yf
import statsmodels.api as sm
import pandas as pd
import numpy as np
import os

In [ ]:
RF_ANNUAL    = 0.065          # RBI repo-rate proxy (6.5% p.a.)
RF_DAILY     = RF_ANNUAL / 252
MIN_OBS_Q    = 40             # minimum trading days for quarterly windows
MIN_OBS_A    = 200            # minimum trading days for annual windows
FILING_WIN_Q = 63             # trading days forward from quarterly filing date
FILING_WIN_A = 252            # trading days forward from annual earliest filing date
BASE = Path(os.getcwd())


Q_CALENDAR = {
    'Q1FY23': ('2022-04-01', '2022-06-30'),
    'Q2FY23': ('2022-07-01', '2022-09-30'),
    'Q3FY23': ('2022-10-01', '2022-12-31'),
    'Q4FY23': ('2023-01-01', '2023-03-31'),
    'Q1FY24': ('2023-04-01', '2023-06-30'),
    'Q2FY24': ('2023-07-01', '2023-09-30'),
    'Q3FY24': ('2023-10-01', '2023-12-31'),
    'Q4FY24': ('2024-01-01', '2024-03-31'),
    'Q1FY25': ('2024-04-01', '2024-06-30'),
    'Q2FY25': ('2024-07-01', '2024-09-30'),
    'Q3FY25': ('2024-10-01', '2024-12-31'),
    'Q4FY25': ('2025-01-01', '2025-03-31'),
}


ANNUAL_FY = {
    'FY23': ('2022-04-01', '2023-03-31'),
    'FY24': ('2023-04-01', '2024-03-31'),
    'FY25': ('2024-04-01', '2025-03-31'),
}

INDIAN_FY_END = {
    'FY23': pd.Timestamp('2023-03-31'),
    'FY24': pd.Timestamp('2024-03-31'),
    'FY25': pd.Timestamp('2025-03-31'),
}

In [ ]:
industry_map = pd.read_excel(BASE / 'data' / 'industry_map.xlsx')
industry_map = industry_map.dropna(subset=['NSE Symbol']).copy()
industry_map['BSE Code'] = pd.to_numeric(industry_map['BSE Code'], errors='coerce').astype('Int64')

tickers_ns      = [f"{s}.NS" for s in industry_map['NSE Symbol']]
sym_to_bse      = dict(zip(industry_map['NSE Symbol'], industry_map['BSE Code']))
sym_to_industry = dict(zip(industry_map['NSE Symbol'], industry_map['Industry']))
sym_to_sector   = dict(zip(industry_map['NSE Symbol'], industry_map['Sector']))


fdb = pd.read_csv(BASE / 'data' / 'filing_dates_db.csv')
fdb['BSE_Code']    = pd.to_numeric(fdb['BSE_Code'], errors='coerce').astype('Int64')
fdb['Filing_Date'] = pd.to_datetime(fdb['Filing_Date'])

_map = industry_map[['BSE Code', 'NSE Symbol', 'Industry', 'Sector']]

# Quarterly: one filing date per (firm, Q_FY) — filter to analysis quarters
filing_q = (
    fdb[fdb['Q_FY'].isin(Q_CALENDAR.keys())][['BSE_Code', 'Q_FY', 'Filing_Date']]
    .merge(_map, left_on='BSE_Code', right_on='BSE Code', how='inner')
    .drop(columns='BSE Code')
    .reset_index(drop=True)
)

# Annual: earliest filing date per (firm, FY)
def earliest_filing_per_fy(fdb, fy_list):
    rows = []
    for fy in fy_list:
        yr  = int(fy.replace('FY', ''))
        qs  = [f'Q{q}FY{yr}' for q in range(1, 5)]
        sub = fdb[fdb['Q_FY'].isin(qs)]
        if sub.empty:
            continue
        e = sub.groupby('BSE_Code')['Filing_Date'].min().reset_index()
        e['FY'] = fy
        rows.append(e)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

filing_a = (
    earliest_filing_per_fy(fdb, list(ANNUAL_FY))
    .merge(_map, left_on='BSE_Code', right_on='BSE Code', how='inner')
    .drop(columns='BSE Code')
    .reset_index(drop=True)
)

print(f"Companies       : {len(tickers_ns)}")
print(f"Q filing rows   : {len(filing_q):,}  ({filing_q['Q_FY'].nunique()} quarters, {filing_q['NSE Symbol'].nunique()} firms)")
print(f"Annual filing rows: {len(filing_a):,}  ({filing_a['FY'].nunique()} FYs, {filing_a['NSE Symbol'].nunique()} firms)")

Companies       : 247
Q filing rows   : 2,911  (12 quarters, 247 firms)
Annual filing rows: 741  (3 FYs, 247 firms)


In [ ]:
print("Downloading Nifty 500 (^CRSLDX)...")
nifty_raw = yf.download('^CRSLDX', start='2022-01-01', end='2026-04-30',
                        auto_adjust=True, progress=False)
nifty_ret       = nifty_raw['Close'].squeeze().pct_change().dropna()
nifty_ret.name  = 'market'
print(f"  Nifty 500: {len(nifty_ret)} trading days")

print("\nDownloading stock prices (batch)...")
prices_raw = yf.download(tickers_ns, start='2022-01-01', end='2026-04-30',
                         auto_adjust=True, progress=True)['Close']

valid     = prices_raw.columns[prices_raw.isna().mean() < 0.5]
stock_ret = prices_raw[valid].pct_change().dropna(how='all')

log_mkt   = np.log1p(nifty_ret)
log_stock = np.log1p(stock_ret)
trading_days = log_mkt.index   # market calendar — used for forward windows

print(f"\nValid tickers : {len(valid)} / {len(tickers_ns)}")
print(f"Date range    : {log_mkt.index[0].date()} \u2192 {log_mkt.index[-1].date()}")
print(f"Trading days  : {len(log_mkt)}")

  Nifty 500: 1043 trading days



[****                   9%                       ]  21 of 247 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ISEC.NS"}}}
[****                   9%                       ]  23 of 247 completed$ISEC.NS: possibly delisted; no timezone found
[***************       31%                       ]  76 of 247 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SUVENPHARMA.NS"}}}
[***************       32%                       ]  78 of 247 completed$SUVENPHARMA.NS: possibly delisted; no timezone found
[*****************     36%                       ]  90 of 247 completed$PEL.NS: possibly delisted; no timezone found
[********************* 44%                       ]  108 of 247 completed$ZOMATO.NS: possibly delisted; no timezone found
[**********************50%                       ]  124 of 247 completed$GET&D.NS: possibly delisted; no timez


Valid tickers : 239 / 247
Date range    : 2022-01-04 → 2026-04-06
Trading days  : 1043


In [ ]:
def fwd_window(filing_dt, n_days):
    """Next n_days trading days strictly after filing_dt."""
    return trading_days[trading_days > pd.Timestamp(filing_dt)][:n_days]

def add_adj_returns(df, period_col):
    """Append IAR (industry-adj) and SAR (sector-adj) columns."""
    for grp_col, new_col in [('Industry', 'iar_pct'), ('Sector', 'sar_pct')]:
        med = (df.groupby([grp_col, period_col])['cum_return']
                 .median().reset_index()
                 .rename(columns={'cum_return': '_med'}))
        df = df.merge(med, on=[grp_col, period_col], how='left')
        df[new_col] = df['cum_return'] - df['_med']
        df = df.drop(columns='_med')
    return df

def compute_vol(stk_ex, mkt_ex):
    """Returns (beta, total_vol, downside_vol) from excess-return series."""
    try:
        res   = sm.OLS(stk_ex.values, sm.add_constant(mkt_ex.values)).fit()
        resid = res.resid
        beta  = float(res.params[1])
    except Exception:
        resid = np.array([])
        beta  = np.nan
    total_vol    = float(stk_ex.std(ddof=1))
    downside_vol = float(resid[resid < 0].std(ddof=1))  if len(resid) > 0 and (resid < 0).any() else np.nan
    return beta, total_vol, downside_vol

def compute_capm(stk_ex, mkt_ex):
    """Returns (alpha_daily, beta, r2, p_alpha) from excess-return series."""
    try:
        res     = sm.OLS(stk_ex.values, sm.add_constant(mkt_ex.values)).fit()
        alpha_d = float(res.params[0])
        beta    = float(res.params[1])
        r2      = float(res.rsquared)
        p_alpha = float(res.pvalues[0])
    except Exception:
        alpha_d = beta = r2 = p_alpha = np.nan
    return alpha_d, beta, r2, p_alpha

print('Helpers defined.')

Helpers defined.


---
## Quarterly Metrics — Filing-Date Window (63 trading days)

In [ ]:
qmar_records = []

for _, row in filing_q.iterrows():
    sym       = row['NSE Symbol']
    ticker    = sym + '.NS'
    q_fy      = row['Q_FY']
    filing_dt = row['Filing_Date']
    if ticker not in log_stock.columns:
        continue
    window = fwd_window(filing_dt, FILING_WIN_Q)
    if len(window) < MIN_OBS_Q:
        continue
    aligned = pd.concat([log_stock.loc[window, ticker].rename('stk'),
                         log_mkt.loc[window].rename('mkt')], axis=1).dropna()
    if len(aligned) < MIN_OBS_Q:
        continue
    cum_stk = aligned['stk'].sum()
    cum_mkt = aligned['mkt'].sum()
    mar     = cum_stk - cum_mkt
    qmar_records.append({
        'NSE Symbol'    : sym,
        'BSE Code'      : int(row['BSE_Code']),
        'Industry'      : row['Industry'],
        'Sector'        : row['Sector'],
        'Q_FY'          : q_fy,
        'Filing_Date'   : filing_dt.date(),
        'Window_Start'  : aligned.index[0].date(),
        'Window_End'    : aligned.index[-1].date(),
        'n_trading_days': len(aligned),
        'cum_return'    : np.expm1(cum_stk),
        'cum_mkt_return': np.expm1(cum_mkt),
        'mar_log'       : mar,
        'mar_pct'       : np.expm1(mar),
    })

qmar_df = add_adj_returns(pd.DataFrame(qmar_records), 'Q_FY')

out = BASE / 'data' / 'quarterly_returns_from_filing.csv'
qmar_df.to_csv(out, index=False)
print(f'Rows: {len(qmar_df):,}  Firms: {qmar_df["NSE Symbol"].nunique()}  Quarters: {qmar_df["Q_FY"].nunique()}')
print('\nMAR mean/std by quarter (%):')
print(qmar_df.groupby('Q_FY')['mar_pct'].agg(['mean','std']).mul(100).round(2))

Rows: 2,807  Firms: 239  Quarters: 12

MAR mean/std by quarter (%):
         mean    std
Q_FY                
Q1FY23  -0.64  18.24
Q1FY24   6.10  20.06
Q1FY25   0.49  13.73
Q2FY23   0.99  14.01
Q2FY24   3.46  19.57
Q2FY25   1.10  13.83
Q3FY23   6.79  16.95
Q3FY24   6.07  21.13
Q3FY25   7.07  15.09
Q4FY23  10.90  21.65
Q4FY24   3.77  19.73
Q4FY25  -1.33  11.18


In [ ]:
qalpha_records = []

for _, row in filing_q.iterrows():
    sym       = row['NSE Symbol']
    ticker    = sym + '.NS'
    q_fy      = row['Q_FY']
    filing_dt = row['Filing_Date']
    if ticker not in log_stock.columns:
        continue
    window = fwd_window(filing_dt, FILING_WIN_Q)
    if len(window) < MIN_OBS_Q:
        continue
    aligned = pd.concat([log_stock.loc[window, ticker].rename('stk'),
                         log_mkt.loc[window].rename('mkt')], axis=1).dropna()
    if len(aligned) < MIN_OBS_Q:
        continue
    stk_ex = aligned['stk'] - RF_DAILY
    mkt_ex = aligned['mkt'] - RF_DAILY
    alpha_d, beta, r2, p_alpha = compute_capm(stk_ex, mkt_ex)
    qalpha_records.append({
        'NSE Symbol'          : sym,
        'BSE Code'            : int(row['BSE_Code']),
        'Industry'            : row['Industry'],
        'Sector'              : row['Sector'],
        'Q_FY'                : q_fy,
        'Filing_Date'         : filing_dt.date(),
        'Window_Start'        : aligned.index[0].date(),
        'Window_End'          : aligned.index[-1].date(),
        'n_obs'               : len(aligned),
        'capm_alpha_daily'    : alpha_d,
        'capm_beta'           : beta,
        'capm_alpha_quarterly': np.expm1(alpha_d * len(aligned)) if not np.isnan(alpha_d) else np.nan,
        'capm_alpha_pval'     : p_alpha,
        'capm_r2'             : r2,
    })

qalpha_df = pd.DataFrame(qalpha_records)

out = BASE / 'data' / 'quarterly_alpha_from_filing.csv'
qalpha_df.to_csv(out, index=False)
print(f'Rows: {len(qalpha_df):,}  Firms: {qalpha_df["NSE Symbol"].nunique()}')
print('\nCAPM alpha mean/std by quarter (%):')
print(qalpha_df.groupby('Q_FY')['capm_alpha_quarterly'].agg(['mean','std']).mul(100).round(2))

Rows: 2,807  Firms: 239

CAPM alpha mean/std by quarter (%):
         mean    std
Q_FY                
Q1FY23  -0.58  18.11
Q1FY24   5.34  18.68
Q1FY25   2.03  14.60
Q2FY23   0.74  13.53
Q2FY24   2.47  18.87
Q2FY25   0.57  13.58
Q3FY23   8.84  18.52
Q3FY24   5.77  20.15
Q3FY25   6.98  15.08
Q4FY23  10.60  20.87
Q4FY24   3.73  19.72
Q4FY25  -1.41  11.12


In [10]:
# Quarterly Volatility  (filing-date start, 63 td)
qvol_filing_records = []

for _, row in filing_q.iterrows():
    sym       = row['NSE Symbol']
    ticker    = sym + '.NS'
    q_fy      = row['Q_FY']
    filing_dt = row['Filing_Date']
    if ticker not in log_stock.columns:
        continue
    window = fwd_window(filing_dt, FILING_WIN_Q)
    if len(window) < MIN_OBS_Q:
        continue
    aligned = pd.concat([log_stock.loc[window, ticker].rename('stk'),
                         log_mkt.loc[window].rename('mkt')], axis=1).dropna()
    if len(aligned) < MIN_OBS_Q:
        continue
    stk_ex = aligned['stk'] - RF_DAILY
    mkt_ex = aligned['mkt'] - RF_DAILY
    beta, total_vol, downside_vol = compute_vol(stk_ex, mkt_ex)
    qvol_filing_records.append({
        'NSE Symbol'  : sym,
        'BSE Code'    : int(row['BSE_Code']),
        'Industry'    : row['Industry'],
        'Sector'      : row['Sector'],
        'Q_FY'        : q_fy,
        'Filing_Date' : filing_dt.date(),
        'Window_Start': aligned.index[0].date(),
        'Window_End'  : aligned.index[-1].date(),
        'n_obs'       : len(aligned),
        'beta'        : beta,
        'total_vol'   : total_vol,
        'downside_vol': downside_vol,
    })

qvol_filing_df = pd.DataFrame(qvol_filing_records)

out = BASE / 'data' / 'quarterly_vol_from_filing.csv'
qvol_filing_df.to_csv(out, index=False)
print(f'Rows: {len(qvol_filing_df):,}')
print('\nMean total_vol by quarter (daily):')
print(qvol_filing_df.groupby('Q_FY')['total_vol'].mean().round(5))

Rows: 2,807

Mean total_vol by quarter (daily):
Q_FY
Q1FY23    0.01921
Q1FY24    0.02086
Q1FY25    0.02310
Q2FY23    0.01929
Q2FY24    0.02407
Q2FY25    0.02494
Q3FY23    0.01738
Q3FY24    0.02527
Q3FY25    0.01939
Q4FY23    0.01938
Q4FY24    0.02131
Q4FY25    0.01735
Name: total_vol, dtype: float64


In [11]:
# Quarterly Volatility (calendar quarter dates)
qvol_cal_records = []

for q_fy, (start, end) in Q_CALENDAR.items():
    mkt_q = log_mkt.loc[start:end]
    if len(mkt_q) < MIN_OBS_Q:
        continue
    for ticker in valid:
        sym   = ticker.replace('.NS', '')
        stk_q = log_stock.loc[start:end, ticker].squeeze()
        aligned = pd.concat([stk_q.rename('stk'), mkt_q.rename('mkt')], axis = 1).dropna()
        if len(aligned) < MIN_OBS_Q:
            continue
        stk_ex = aligned['stk'] - RF_DAILY
        mkt_ex = aligned['mkt'] - RF_DAILY
        beta, total_vol, downside_vol = compute_vol(stk_ex, mkt_ex)
        qvol_cal_records.append({
            'NSE Symbol'  : sym,
            'BSE Code'    : sym_to_bse.get(sym),
            'Industry'    : sym_to_industry.get(sym),
            'Sector'      : sym_to_sector.get(sym),
            'Q_FY'        : q_fy,
            'Window_Start': aligned.index[0].date(),
            'Window_End'  : aligned.index[-1].date(),
            'n_obs'       : len(aligned),
            'beta'        : beta,
            'total_vol'   : total_vol,
            'downside_vol': downside_vol,
        })

qvol_cal_df = pd.DataFrame(qvol_cal_records)

out = BASE / 'data' / 'quarterly_vol_from_qdate.csv'
qvol_cal_df.to_csv(out, index=False)
print(f'Rows: {len(qvol_cal_df):,}  Firms: {qvol_cal_df["NSE Symbol"].nunique()}')
print('\nMean total_vol by quarter (daily):')
print(qvol_cal_df.groupby('Q_FY')['total_vol'].mean().round(5))

Rows: 2,828  Firms: 239

Mean total_vol by quarter (daily):
Q_FY
Q1FY23    0.02479
Q1FY24    0.01746
Q1FY25    0.02542
Q2FY23    0.02040
Q2FY24    0.01926
Q2FY25    0.02092
Q3FY23    0.02005
Q3FY24    0.02011
Q3FY25    0.02223
Q4FY23    0.01911
Q4FY24    0.02405
Q4FY25    0.02566
Name: total_vol, dtype: float64


---
## Annual Metrics — Filing-Date Window (252 trading days)

In [12]:
# Annual MAR + IAR + SAR  (earliest FY filing start, 252 td)
amar_records = []

for _, row in filing_a.iterrows():
    sym       = row['NSE Symbol']
    ticker    = sym + '.NS'
    fy        = row['FY']
    filing_dt = row['Filing_Date']
    if ticker not in log_stock.columns:
        continue
    window = fwd_window(filing_dt, FILING_WIN_A)
    if len(window) < MIN_OBS_A:
        continue
    aligned = pd.concat([log_stock.loc[window, ticker].rename('stk'),
                         log_mkt.loc[window].rename('mkt')], axis=1).dropna()
    if len(aligned) < MIN_OBS_A:
        continue
    cum_stk = aligned['stk'].sum()
    cum_mkt = aligned['mkt'].sum()
    mar     = cum_stk - cum_mkt
    amar_records.append({
        'NSE Symbol'    : sym,
        'BSE Code'      : int(row['BSE_Code']),
        'Industry'      : row['Industry'],
        'Sector'        : row['Sector'],
        'FY'            : fy,
        'Filing_Date'   : filing_dt.date(),
        'Window_Start'  : aligned.index[0].date(),
        'Window_End'    : aligned.index[-1].date(),
        'n_trading_days': len(aligned),
        'cum_return'    : np.expm1(cum_stk),
        'cum_mkt_return': np.expm1(cum_mkt),
        'mar_log'       : mar,
        'mar_pct'       : np.expm1(mar),
    })

amar_df = add_adj_returns(pd.DataFrame(amar_records), 'FY')

out = BASE / 'data' / 'annual_returns_from_filing.csv'
amar_df.to_csv(out, index=False)
print(f'Rows: {len(amar_df):,}  Firms: {amar_df["NSE Symbol"].nunique()}  FYs: {sorted(amar_df["FY"].unique())}')
print('\nMAR mean/std by FY (%):')
print(amar_df.groupby('FY')['mar_pct'].agg(['mean','std']).mul(100).round(2))

Rows: 714  Firms: 239  FYs: ['FY23', 'FY24', 'FY25']

MAR mean/std by FY (%):
       mean    std
FY                
FY23  19.38  43.21
FY24  18.53  42.59
FY25   2.55  26.57


In [13]:
# Annual CAPM Alpha  (earliest FY filing start, 252 td)
aalpha_records = []

for _, row in filing_a.iterrows():
    sym       = row['NSE Symbol']
    ticker    = sym + '.NS'
    fy        = row['FY']
    filing_dt = row['Filing_Date']
    if ticker not in log_stock.columns:
        continue
    window = fwd_window(filing_dt, FILING_WIN_A)
    if len(window) < MIN_OBS_A:
        continue
    aligned = pd.concat([log_stock.loc[window, ticker].rename('stk'),
                         log_mkt.loc[window].rename('mkt')], axis=1).dropna()
    if len(aligned) < MIN_OBS_A:
        continue
    stk_ex = aligned['stk'] - RF_DAILY
    mkt_ex = aligned['mkt'] - RF_DAILY
    alpha_d, beta, r2, p_alpha = compute_capm(stk_ex, mkt_ex)
    aalpha_records.append({
        'NSE Symbol'        : sym,
        'BSE Code'          : int(row['BSE_Code']),
        'Industry'          : row['Industry'],
        'Sector'            : row['Sector'],
        'FY'                : fy,
        'Filing_Date'       : filing_dt.date(),
        'Window_Start'      : aligned.index[0].date(),
        'Window_End'        : aligned.index[-1].date(),
        'n_obs'             : len(aligned),
        'capm_alpha_daily'  : alpha_d,
        'capm_beta'         : beta,
        'capm_alpha_annual' : np.expm1(alpha_d * len(aligned)) if not np.isnan(alpha_d) else np.nan,
        'capm_alpha_pval'   : p_alpha,
        'capm_r2'           : r2,
    })

aalpha_df = pd.DataFrame(aalpha_records)

out = BASE / 'data' / 'annual_alpha_from_filing.csv'
aalpha_df.to_csv(out, index=False)
print(f'Rows: {len(aalpha_df):,}  Firms: {aalpha_df["NSE Symbol"].nunique()}')
print('\nAnnual CAPM alpha mean/std by FY (%):')
print(aalpha_df.groupby('FY')['capm_alpha_annual'].agg(['mean','std']).mul(100).round(2))

Rows: 714  Firms: 239

Annual CAPM alpha mean/std by FY (%):
       mean    std
FY                
FY23  19.28  42.17
FY24  15.62  38.90
FY25   3.51  27.12


In [14]:
# Annual Volatility  (earliest FY filing start, 252 td)
avol_filing_records = []

for _, row in filing_a.iterrows():
    sym       = row['NSE Symbol']
    ticker    = sym + '.NS'
    fy        = row['FY']
    filing_dt = row['Filing_Date']
    if ticker not in log_stock.columns:
        continue
    window = fwd_window(filing_dt, FILING_WIN_A)
    if len(window) < MIN_OBS_A:
        continue
    aligned = pd.concat([log_stock.loc[window, ticker].rename('stk'),
                         log_mkt.loc[window].rename('mkt')], axis=1).dropna()
    if len(aligned) < MIN_OBS_A:
        continue
    stk_ex = aligned['stk'] - RF_DAILY
    mkt_ex = aligned['mkt'] - RF_DAILY
    beta, total_vol, downside_vol = compute_vol(stk_ex, mkt_ex)
    avol_filing_records.append({
        'NSE Symbol'  : sym,
        'BSE Code'    : int(row['BSE_Code']),
        'Industry'    : row['Industry'],
        'Sector'      : row['Sector'],
        'FY'          : fy,
        'Filing_Date' : filing_dt.date(),
        'Window_Start': aligned.index[0].date(),
        'Window_End'  : aligned.index[-1].date(),
        'n_obs'       : len(aligned),
        'beta'        : beta,
        'total_vol'   : total_vol,
        'downside_vol': downside_vol,
    })

avol_filing_df = pd.DataFrame(avol_filing_records)

out = BASE / 'data' / 'annual_vol_from_filing.csv'
avol_filing_df.to_csv(out, index=False)
print(f'Rows: {len(avol_filing_df):,}  Firms: {avol_filing_df["NSE Symbol"].nunique()}')
print('\nMean total_vol by FY (daily):')
print(avol_filing_df.groupby('FY')['total_vol'].mean().round(5))

Rows: 714  Firms: 239

Mean total_vol by FY (daily):
FY
FY23    0.01945
FY24    0.02349
FY25    0.02195
Name: total_vol, dtype: float64


In [15]:
# Annual Volatility  (full FY calendar window: Apr–Mar)
avol_fy_records = []

for fy, (start, end) in ANNUAL_FY.items():
    mkt_fy = log_mkt.loc[start:end]
    if len(mkt_fy) < MIN_OBS_A:
        continue
    for ticker in valid:
        sym   = ticker.replace('.NS', '')
        stk_w = log_stock.loc[start:end, ticker].squeeze()
        aligned = pd.concat([stk_w.rename('stk'), mkt_fy.rename('mkt')], axis=1).dropna()
        if len(aligned) < MIN_OBS_A:
            continue
        stk_ex = aligned['stk'] - RF_DAILY
        mkt_ex = aligned['mkt'] - RF_DAILY
        beta, total_vol, downside_vol = compute_vol(stk_ex, mkt_ex)
        avol_fy_records.append({
            'NSE Symbol'  : sym,
            'BSE Code'    : sym_to_bse.get(sym),
            'Industry'    : sym_to_industry.get(sym),
            'Sector'      : sym_to_sector.get(sym),
            'FY'          : fy,
            'Window_Start': aligned.index[0].date(),
            'Window_End'  : aligned.index[-1].date(),
            'n_obs'       : len(aligned),
            'beta'        : beta,
            'total_vol'   : total_vol,
            'downside_vol': downside_vol,
        })

avol_fy_df = pd.DataFrame(avol_fy_records)

out = BASE / 'data' / 'annual_vol_from_fydate.csv'
avol_fy_df.to_csv(out, index=False)
print(f'Rows: {len(avol_fy_df):,}  Firms: {avol_fy_df["NSE Symbol"].nunique()}')
print('\nMean total_vol by FY (daily):')
print(avol_fy_df.groupby('FY')['total_vol'].mean().round(5))

Rows: 704  Firms: 239

Mean total_vol by FY (daily):
FY
FY23    0.02165
FY24    0.02067
FY25    0.02396
Name: total_vol, dtype: float64


---
## Annual ROE

In [16]:
# Annual ROE from yfinance  (FY Apr–Mar + CG filing date tagged)
roe_records = []
failed_roe  = []

NI_ROWS = ['Net Income', 'NetIncome', 'Net Income Common Stockholders']
EQ_ROWS = ['Stockholders Equity', 'Total Stockholder Equity',
           'Common Stock Equity', 'Total Equity Gross Minority Interest']

for i, ticker in enumerate(tickers_ns):
    sym = ticker.replace('.NS', '')
    try:
        t   = yf.Ticker(ticker)
        inc = t.financials
        bal = t.balance_sheet
        if inc is None or bal is None or inc.empty or bal.empty:
            failed_roe.append(sym); continue
        ni_row = next((r for r in NI_ROWS if r in inc.index), None)
        eq_row = next((r for r in EQ_ROWS if r in bal.index), None)
        if ni_row is None or eq_row is None:
            failed_roe.append(sym); continue
        ni = inc.loc[ni_row].dropna()
        eq = bal.loc[eq_row].dropna()
        for fy, fy_end in INDIAN_FY_END.items():
            win_s = fy_end - pd.DateOffset(months=9)
            win_e = fy_end + pd.DateOffset(months=3)
            ni_d  = [d for d in ni.index if win_s <= pd.Timestamp(d) <= win_e]
            eq_d  = [d for d in eq.index if win_s <= pd.Timestamp(d) <= win_e]
            if not ni_d or not eq_d:
                continue
            net_income = float(ni[max(ni_d)])
            equity     = float(eq[max(eq_d)])
            if equity == 0 or pd.isna(equity) or pd.isna(net_income):
                continue
            roe_records.append({
                'NSE Symbol': sym,
                'BSE Code'  : sym_to_bse.get(sym),
                'Industry'  : sym_to_industry.get(sym),
                'Sector'    : sym_to_sector.get(sym),
                'FY'        : fy,
                'FY_End'    : fy_end.date(),
                'net_income': net_income,
                'equity'    : equity,
                'roe'       : net_income / abs(equity),
            })
    except Exception:
        failed_roe.append(sym)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(tickers_ns)} done...'); time.sleep(1)

roe_df = pd.DataFrame(roe_records)

# Tag each row with the earliest CG filing date of that FY
roe_df = roe_df.merge(
    filing_a[['NSE Symbol', 'FY', 'Filing_Date']].rename(columns={'Filing_Date': 'CG_Filing_Date'}),
    on=['NSE Symbol', 'FY'], how='left'
)

out = BASE / 'data' / 'annual_roe.csv'
roe_df.to_csv(out, index=False)
print(f'Rows: {len(roe_df):,}  Companies: {roe_df["NSE Symbol"].nunique()}  FYs: {sorted(roe_df["FY"].unique())}')
print(f'Failed: {len(failed_roe)}')
print('\nROE by FY:')
print(roe_df.groupby('FY')['roe'].describe().round(3))

  50/247 done...
  100/247 done...
  150/247 done...
  200/247 done...
Rows: 723  Companies: 242  FYs: ['FY23', 'FY24', 'FY25']
Failed: 5

ROE by FY:
      count   mean    std    min    25%    50%    75%    max
FY                                                          
FY23  240.0  0.156  0.153 -0.394  0.093  0.141  0.201  1.260
FY24  241.0  0.170  0.280 -0.300  0.098  0.145  0.207  4.093
FY25  242.0  0.165  0.160 -0.389  0.102  0.142  0.193  1.569
